# 03 — XGBoost and Baseline Comparison

**Goal:** See whether gradient-boosted trees beat the logistic-regression baseline on AUC, and by how much.

**What we expect:** XGBoost typically gains a few points of AUC over LR on this dataset because it captures nonlinearities (e.g. `tenure × Contract` interactions) automatically. If the gain is tiny, we'd ship LR — it's more interpretable and cheaper to run.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_clean
from src.features import make_features, train_test_split_scaled
from src.model import train_logistic_regression, train_xgboost
from src.evaluation import (
    classification_metrics,
    feature_importance_df,
    plot_confusion,
    plot_roc,
)

## 1. Data

We fit **both** models against the same train/test split so comparisons are apples-to-apples. Scaling is only needed for LR; XGBoost is invariant to monotonic transforms of numeric features, but applying it doesn't hurt and keeps the feature matrix identical.

In [ ]:
df = load_clean()
X, y = make_features(df)
X_train, X_test, y_train, y_test = train_test_split_scaled(X, y, scale=True)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 2. Fit both models

In [ ]:
lr = train_logistic_regression(X_train, y_train)
xgb = train_xgboost(X_train, y_train)

## 3. Head-to-head metrics

In [ ]:
results = {}
for name, mdl in [("logistic_regression", lr), ("xgboost", xgb)]:
    y_pred = mdl.predict(X_test)
    y_proba = mdl.predict_proba(X_test)[:, 1]
    results[name] = classification_metrics(y_test, y_pred, y_proba)

pd.DataFrame(results).round(3)

## 4. ROC overlay — same plot, both models

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(y_test, lr.predict_proba(X_test)[:, 1], label="Logistic Regression", ax=ax)
plot_roc(y_test, xgb.predict_proba(X_test)[:, 1], label="XGBoost", ax=ax)
plt.show()

## 5. XGBoost confusion matrix + feature importance

In [ ]:
y_pred_xgb = xgb.predict(X_test)
plot_confusion(y_test, y_pred_xgb)
plt.show()

In [ ]:
importances = feature_importance_df(xgb, X_train.columns.tolist())
importances.head(15)

In [ ]:
top = importances.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top["feature"], top["importance"], color="seagreen")
ax.set_title("XGBoost — top 12 feature importances")
ax.set_xlabel("gain-weighted importance")
plt.tight_layout()
plt.show()

## 6. Takeaways and handoff to Week 2

- **Winning model:** whichever has higher test AUC in the comparison table above (typically XGBoost by ~0.01–0.03 on this dataset).
- **Feature-importance agreement:** `Contract`, `tenure`, and `InternetService` should dominate both models. If they don't, investigate data leakage or a featurization bug.
- **Decision threshold is still 0.5** — that's a modeling default, not a business decision. Week 2 (`04_cost_analysis.ipynb`) will derive the optimal threshold from intervention cost and expected retention lift.